In [161]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [162]:
movies_df = pd.read_csv("/content/movies.csv",
                      usecols=['movieId', 'title'],dtype={'movieId': 'int32', 'title':'str'})

In [163]:
ratings_df = pd.read_csv("/content/ratings.csv",
                      usecols=['userId', 'movieId', 'rating'],dtype={'userId': 'int32', 'movieId':'int32', 'rating':'float32'})

In [164]:
movies_df.head()

,movieId,title
0,1,Toy Story (1995)
1,2,Jumanji (1995)
2,3,Grumpier Old Men (1995)
3,4,Waiting to Exhale (1995)
4,5,Father of the Bride Part II (1995)


In [165]:
movies_df.shape

(9742, 2)

In [166]:
ratings_df.head()

,userId,movieId,rating
0,1,1,4.000
1,1,3,4.000
2,1,6,4.000
3,1,47,5.000
4,1,50,5.000


In [167]:
ratings_df.shape

(100836, 3)

In [168]:
df = pd.merge(ratings_df, movies_df, on = 'movieId')
df

,userId,movieId,rating,title
0,1,1,4.000,Toy Story (1995)
1,1,3,4.000,Grumpier Old Men (1995)
2,1,6,4.000,Heat (1995)
3,1,47,5.000,Seven (a.k.a. Se7en) (1995)
4,1,50,5.000,"Usual Suspects, The (1995)"
...,...,...,...,...
100831,610,166534,4.000,Split (2017)
100832,610,168248,5.000,John Wick: Chapter Two (2017)
100833,610,168250,5.000,Get Out (2017)
100834,610,168252,5.000,Logan (2017)


In [169]:
combine_movies_rating = df.dropna(axis = 0, subset = ['title'])
movie_ratingCount = (combine_movies_rating.groupby(by = ['title'])['rating'].count().reset_index().rename(columns = {'rating': 'totalRatingCount'})
[['title', 'totalRatingCount']])
movie_ratingCount

,title,totalRatingCount
0,'71 (2014),1
1,'Hellboy': The Seeds of Creation (2004),1
2,'Round Midnight (1986),2
3,'Salem's Lot (2004),1
4,'Til There Was You (1997),2
...,...,...
9714,eXistenZ (1999),22
9715,xXx (2002),24
9716,xXx: State of the Union (2005),5
9717,¡Three Amigos! (1986),26


In [170]:
ratings_with_totalRatingCount = combine_movies_rating.merge(movie_ratingCount, left_on = 'title', right_on = 'title', how = 'left')
ratings_with_totalRatingCount.head()

,userId,movieId,rating,title,totalRatingCount
0,1,1,4.000,Toy Story (1995),215
1,1,3,4.000,Grumpier Old Men (1995),52
2,1,6,4.000,Heat (1995),102
3,1,47,5.000,Seven (a.k.a. Se7en) (1995),203
4,1,50,5.000,"Usual Suspects, The (1995)",204


In [171]:
pd.set_option('display.float_format', lambda x: '%.3f' % x)
print(movie_ratingCount['totalRatingCount'].describe())

count   9719.000
mean      10.375
std       22.406
min        1.000
25%        1.000
50%        3.000
75%        9.000
max      329.000
Name: totalRatingCount, dtype: float64


In [172]:
popularity_threshold = 100
rating_popular_movie = ratings_with_totalRatingCount.query('totalRatingCount >= @popularity_threshold')
rating_popular_movie.head()

,userId,movieId,rating,title,totalRatingCount
0,1,1,4.000,Toy Story (1995),215
2,1,6,4.000,Heat (1995),102
3,1,47,5.000,Seven (a.k.a. Se7en) (1995),203
4,1,50,5.000,"Usual Suspects, The (1995)",204
7,1,110,4.000,Braveheart (1995),237


In [173]:
rating_popular_movie.shape

(20188, 5)

In [174]:
# First Lets create a Pivot Matrix

movie_features_df = rating_popular_movie.pivot_table(index = 'title', columns = 'userId', values = 'rating').fillna(0)
movie_features_df.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
title,,,,,,,,,,,,,,,,,,,,,
2001: A Space Odyssey (1968),0.000,0.000,0.000,0.000,0.000,0.000,4.000,0.000,0.000,0.000,...,0.000,0.000,5.000,0.000,0.000,5.000,0.000,3.000,0.000,4.500
Ace Ventura: Pet Detective (1994),0.000,0.000,0.000,0.000,3.000,3.000,0.000,0.000,0.000,0.000,...,0.000,2.000,0.000,2.000,0.000,0.000,0.000,3.500,0.000,3.000
Aladdin (1992),0.000,0.000,0.000,4.000,4.000,5.000,3.000,0.000,0.000,4.000,...,0.000,0.000,0.000,3.000,3.500,0.000,0.000,3.000,0.000,0.000
Alien (1979),4.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,5.000,0.000,0.000,4.000,3.000,4.000,0.000,4.500
Aliens (1986),0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,4.000,0.000,0.000,3.500,0.000,4.500,0.000,5.000


In [175]:
from scipy.sparse import csr_matrix

movie_features_df_matrix = csr_matrix(movie_features_df.values)

from sklearn.neighbors import NearestNeighbors

model_knn = NearestNeighbors(metric = 'cosine', algorithm = 'brute')
model_knn.fit(movie_features_df_matrix)

NearestNeighbors(algorithm='brute', metric='cosine')

In [176]:
movie_features_df.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
title,,,,,,,,,,,,,,,,,,,,,
2001: A Space Odyssey (1968),0.000,0.000,0.000,0.000,0.000,0.000,4.000,0.000,0.000,0.000,...,0.000,0.000,5.000,0.000,0.000,5.000,0.000,3.000,0.000,4.500
Ace Ventura: Pet Detective (1994),0.000,0.000,0.000,0.000,3.000,3.000,0.000,0.000,0.000,0.000,...,0.000,2.000,0.000,2.000,0.000,0.000,0.000,3.500,0.000,3.000
Aladdin (1992),0.000,0.000,0.000,4.000,4.000,5.000,3.000,0.000,0.000,4.000,...,0.000,0.000,0.000,3.000,3.500,0.000,0.000,3.000,0.000,0.000
Alien (1979),4.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,5.000,0.000,0.000,4.000,3.000,4.000,0.000,4.500
Aliens (1986),0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,4.000,0.000,0.000,3.500,0.000,4.500,0.000,5.000


In [177]:
query_index = np.random.choice(movie_features_df.shape[0])
print(query_index)
distances, indices = model_knn.kneighbors(movie_features_df.iloc[query_index,:].values.reshape(1, -1), n_neighbors = 5
                                          )

67


In [178]:
movie_features_df.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
title,,,,,,,,,,,,,,,,,,,,,
2001: A Space Odyssey (1968),0.000,0.000,0.000,0.000,0.000,0.000,4.000,0.000,0.000,0.000,...,0.000,0.000,5.000,0.000,0.000,5.000,0.000,3.000,0.000,4.500
Ace Ventura: Pet Detective (1994),0.000,0.000,0.000,0.000,3.000,3.000,0.000,0.000,0.000,0.000,...,0.000,2.000,0.000,2.000,0.000,0.000,0.000,3.500,0.000,3.000
Aladdin (1992),0.000,0.000,0.000,4.000,4.000,5.000,3.000,0.000,0.000,4.000,...,0.000,0.000,0.000,3.000,3.500,0.000,0.000,3.000,0.000,0.000
Alien (1979),4.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,5.000,0.000,0.000,4.000,3.000,4.000,0.000,4.500
Aliens (1986),0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,4.000,0.000,0.000,3.500,0.000,4.500,0.000,5.000


In [179]:
for i in range(0, len(distances.flatten())):
    if i == 0:
        print('Recommendations for {0}:\n'.format(movie_features_df.index[query_index]))
    else:
        print('{0}: {1}, with distance of {2}:'.format(i, movie_features_df.index[indices.flatten()[i]], distances.flatten()[i]))

Recommendations for Home Alone (1990):

1: Mrs. Doubtfire (1993), with distance of 0.39578694105148315:
2: Lion King, The (1994), with distance of 0.4414706230163574:
3: Pretty Woman (1990), with distance of 0.4444250464439392:
4: Jurassic Park (1993), with distance of 0.4748850464820862:


1. DWI PURBO SAFITRI
2. SALSABILA ZHAFRANY
3. HISNA ABIDAH
4. RIVATUL HASANAH
5. AHMAD IRFAN
6. MOCH. YAZID AL B.